## 3. tf.data로 미니배치 로딩하기

In [3]:
import tensorflow as tf

tf.random.set_seed(1)
x_train = tf.constant([[73., 80., 75.],
                       [93., 88., 93.],
                       [89., 91., 90.],
                       [96., 98., 100.],
                       [73., 66., 70.]], dtype=tf.float32)
y_train = tf.constant([[152.],
                       [185.],
                       [180.],
                       [196.],
                       [142.]], dtype=tf.float32)

dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
dataloader = dataset.shuffle(buffer_size=len(x_train)).batch(2)
model = tf.keras.Sequential([tf.keras.layers.Dense(1, input_shape=(3,))])

optimizer = tf.keras.optimizers.SGD(learning_rate=1e-5)
nb_epochs = 20

for epoch in range(nb_epochs + 1):
    for batch_idx, (x_batch, y_batch) in enumerate(dataloader):
        with tf.GradientTape() as tape:
            pred = model(x_batch, training=True)
            cost = tf.reduce_mean(tf.square(pred - y_batch))

        grads = tape.gradient(cost, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        print(f'Epoch {epoch:4d}/{nb_epochs} Batch {batch_idx+1}/{len(list(dataloader))} Cost: {cost.numpy():.6f}')

new_var = tf.constant([[73., 80., 75.]], dtype=tf.float32)
pred_y = model(new_var)

print('훈련 후 입력이 73, 80, 75일 때의 예측값 :', pred_y.numpy())

Epoch    0/20 Batch 1/3 Cost: 5489.790039
Epoch    0/20 Batch 2/3 Cost: 946.994446
Epoch    0/20 Batch 3/3 Cost: 613.779175
Epoch    1/20 Batch 1/3 Cost: 110.126022
Epoch    1/20 Batch 2/3 Cost: 41.859882
Epoch    1/20 Batch 3/3 Cost: 39.994972
Epoch    2/20 Batch 1/3 Cost: 4.915902
Epoch    2/20 Batch 2/3 Cost: 2.624292
Epoch    2/20 Batch 3/3 Cost: 20.054916
Epoch    3/20 Batch 1/3 Cost: 4.920405
Epoch    3/20 Batch 2/3 Cost: 2.610250
Epoch    3/20 Batch 3/3 Cost: 16.131372
Epoch    4/20 Batch 1/3 Cost: 4.542427
Epoch    4/20 Batch 2/3 Cost: 9.355200
Epoch    4/20 Batch 3/3 Cost: 6.841127
Epoch    5/20 Batch 1/3 Cost: 14.001025
Epoch    5/20 Batch 2/3 Cost: 3.884832
Epoch    5/20 Batch 3/3 Cost: 3.458864
Epoch    6/20 Batch 1/3 Cost: 15.935394
Epoch    6/20 Batch 2/3 Cost: 4.590703
Epoch    6/20 Batch 3/3 Cost: 1.306217
Epoch    7/20 Batch 1/3 Cost: 0.720150
Epoch    7/20 Batch 2/3 Cost: 6.381126
Epoch    7/20 Batch 3/3 Cost: 16.293077
Epoch    8/20 Batch 1/3 Cost: 9.251012
Epoch    

## 5. 커스텀 tf.data.Dataset으로 선형 회귀 구현하기

In [2]:
class CustomDataset(tf.data.Dataset):
    def _generator():
        x_data = [[73., 80., 75.],
                  [93., 88., 93.],
                  [89., 91., 90.],
                  [96., 98., 100.],
                  [73., 66., 70.]]
        y_data = [[152.],
                  [185.],
                  [180.],
                  [196.],
                  [142.]]
        for x, y in zip(x_data, y_data):
            yield tf.constant(x, dtype=tf.float32), tf.constant(y, dtype=tf.float32)
    def __new__(cls):
        return tf.data.Dataset.from_generator(
            cls._generator,
            output_signature=(tf.TensorSpec(shape=(3,), dtype=tf.float32),
                              tf.TensorSpec(shape=(1,), dtype=tf.float32))
        )

dataset = CustomDataset().shuffle(5).batch(2)
model = tf.keras.Sequential([tf.keras.layers.Dense(1, input_shape=(3,))])
optimizer = tf.keras.optimizers.SGD(learning_rate=1e-5)

nb_epochs = 20
for epoch in range(nb_epochs + 1):
    for batch_idx, (x_batch, y_batch) in enumerate(dataset):
        with tf.GradientTape() as tape:

            pred = model(x_batch, training=True)
            cost = tf.reduce_mean(tf.square(pred - y_batch))

        grads = tape.gradient(cost, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        print(f'Epoch {epoch:4d}/{nb_epochs} Batch {batch_idx+1}/{len(list(dataset))} Cost: {cost.numpy():.6f}')

new_var = tf.constant([[73., 80., 75.]], dtype=tf.float32)
pred_y = model(new_var)

print('훈련 후 입력이 73, 80, 75일 때의 예측값 :', pred_y.numpy())

/home/affx/pytorch-nlp-tutorial/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch    0/20 Batch 1/3 Cost: 35349.554688
Epoch    0/20 Batch 2/3 Cost: 5116.751953
Epoch    0/20 Batch 3/3 Cost: 1555.973633
Epoch    1/20 Batch 1/3 Cost: 1160.875732
Epoch    1/20 Batch 2/3 Cost: 163.460068
Epoch    1/20 Batch 3/3 Cost: 199.806702
Epoch    2/20 Batch 1/3 Cost: 26.460323
Epoch    2/20 Batch 2/3 Cost: 0.360275
Epoch    2/20 Batch 3/3 Cost: 20.173990
Epoch    3/20 Batch 1/3 Cost: 5.184627
Epoch    3/20 Batch 2/3 Cost: 7.770328
Epoch    3/20 Batch 3/3 Cost: 0.142958
Epoch    4/20 Batch 1/3 Cost: 0.191231
Epoch    4/20 Batch 2/3 Cost: 6.270853
Epoch    4/20 Batch 3/3 Cost: 5.550390
Epoch    5/20 Batch 1/3 Cost: 2.727273
Epoch    5/20 Batch 2/3 Cost: 10.281546
Epoch    5/20 Batch 3/3 Cost: 0.271946
Epoch    6/20 Batch 1/3 Cost: 2.923369
Epoch    6/20 Batch 2/3 Cost: 3.622322
Epoch    6/20 Batch 3/3 Cost: 6.903689
Epoch    7/20 Batch 1/3 Cost: 5.637673
Epoch    7/20 Batch 2/3 Cost: 4.298746
Epoch    7/20 Batch 3/3 Cost: 0.257146
Epoch    8/20 Batch 1/3 Cost: 1.893936
Epoch

2025-11-28 17:27:12.608831: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch   11/20 Batch 1/3 Cost: 5.037853
Epoch   11/20 Batch 2/3 Cost: 4.285601
Epoch   11/20 Batch 3/3 Cost: 1.014948
Epoch   12/20 Batch 1/3 Cost: 1.310369
Epoch   12/20 Batch 2/3 Cost: 6.335222
Epoch   12/20 Batch 3/3 Cost: 5.256306
Epoch   13/20 Batch 1/3 Cost: 2.571455
Epoch   13/20 Batch 2/3 Cost: 4.310777
Epoch   13/20 Batch 3/3 Cost: 8.340635
Epoch   14/20 Batch 1/3 Cost: 1.439187
Epoch   14/20 Batch 2/3 Cost: 3.850800
Epoch   14/20 Batch 3/3 Cost: 8.473273
Epoch   15/20 Batch 1/3 Cost: 4.612332
Epoch   15/20 Batch 2/3 Cost: 4.449196
Epoch   15/20 Batch 3/3 Cost: 1.519804
Epoch   16/20 Batch 1/3 Cost: 3.092496
Epoch   16/20 Batch 2/3 Cost: 5.771980
Epoch   16/20 Batch 3/3 Cost: 1.241196
Epoch   17/20 Batch 1/3 Cost: 6.046280
Epoch   17/20 Batch 2/3 Cost: 2.873641
Epoch   17/20 Batch 3/3 Cost: 0.422016
Epoch   18/20 Batch 1/3 Cost: 2.451862
Epoch   18/20 Batch 2/3 Cost: 3.784209
Epoch   18/20 Batch 3/3 Cost: 10.331787
Epoch   19/20 Batch 1/3 Cost: 4.452421
Epoch   19/20 Batch 2/3 